# Groundwater Age and Reverse-Capture Zones  
## *Return of the Jedi* Edition

## Overview

This notebook picks up where the base-model and scenario notebooks leave off. Instead of only asking **how much water levels change**, we ask two deeper questions about the aquifer system:

1. **How old is the groundwater moving through different parts of the model?**
2. **For selected pumping wells, where does the model suggest that water may have come from over a chosen travel-time window?**

Those two questions work together. The groundwater-age map gives regional context: some parts of the aquifer behave like fast, young, highly connected pathways, while other areas behave like slower or more isolated parts of the system. The reverse-capture maps then focus on pumping wells and ask which parts of the modeled aquifer are most strongly connected to those wells within selected time windows.

### 🌌 A Water Modeling Parable: The Endor Briefing

*Return of the Jedi* is not just a story about one heroic shot. It is a story about **hidden connections**: the shield generator on Endor protects the Death Star II, the small Ewok villages shape the outcome of a galactic battle, and the most obvious source of danger is not always the most important one.

Groundwater planning has the same lesson. A pumping well may look like a single point on a map, but it is connected to a much larger three-dimensional flow system. Some of those connections are obvious. Others are buried, delayed, or easy to miss. This notebook is our Endor briefing: we use the model to look below the surface, trace connections through time, and separate the strong signals from the background noise.

### What you will learn

1. How to copy the IISG base MODFLOW 6 model into a groundwater-age workspace
2. How to simulate **groundwater age** as a transported model quantity
3. How to run a reverse-GWT experiment that traces a unit signal backward from pumping wells
4. How to map 5-, 10-, and 25-year reverse-capture footprints
5. How to check that the flow information used by the reverse-transport model has actually been reversed
6. How to interpret the maps as participatory-modeling products rather than magic underground fences

### Workshop Context

This is one of three scenario notebooks that build on `intro_base_model.ipynb`:

| Notebook | Scenario / purpose |
|---|---|
| `intro_base_model.ipynb` | Build familiarity with the base IISG groundwater model |
| `new_demands.ipynb` | Explore new pumping demand and well-placement strategies |
| `drought_impacts.ipynb` | Explore reduced recharge and drought response |
| -- **`gw_age.ipynb`** | **Explore groundwater age and reverse-capture connections (this notebook)**  -- |

> **Note — Educational Use Only.** The model uses real-world hydrogeologic concepts and data structures, but it is simplified for workshop use. Results are intended for exploration, comparison, and discussion. They should not be treated as final regulatory boundaries or site-specific engineering decisions without additional review.


## 1. Imports

We start by loading the Python tools used throughout the notebook. Each package has a role in the modeling story:

| Library | Purpose |
|---|---|
| `flopy` | Load, copy, modify, run, and read MODFLOW 6 / GWT models |
| `numpy` | Work with model arrays such as heads, age, concentration, and masks |
| `pandas` | Organize tabular summaries when needed |
| `matplotlib` | Create static maps and figures for discussion |
| `geopandas` | Read and plot county boundaries and other GIS layers |
| `pathlib` | Keep file paths readable and portable |

### 🌌 A Water Modeling Parable: R2-D2 Carries the Plans

In *Return of the Jedi*, the mission only works because the characters can carry, decode, and act on technical information. In this notebook, the Python libraries are our droids and translators. FloPy reads the MODFLOW files, NumPy moves the arrays around, GeoPandas brings in the map context, and Matplotlib turns the hidden model data into something people can actually see and discuss.


In [ ]:
# Standard library
from pathlib import Path

# Scientific stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Geospatial / MODFLOW tools
import geopandas as gpd

import flopy
import flopy.utils.binaryfile as bf # <-- alias a specific package to avoid extra typing later
from flopy.utils.postprocessing import get_specific_discharge # <-- can also pull out functions directly



## 2. Paths and user settings

This cell defines where the notebook expects to find the base model, where it should write the groundwater-age model, where MODFLOW 6 lives, and where the county boundary shapefile is stored.

These settings are intentionally kept near the top. If the notebook is moved to a new computer, this is the first place to check. A groundwater model can be scientifically sound and still fail for the most boring reason in the galaxy: the file path points to the wrong place.

### 🌌 A Water Modeling Parable: Finding the Forest Moon

Before the Rebel fleet can reach Endor, it needs coordinates. Before this notebook can run, Python needs coordinates too: model workspaces, executable paths, and background data. This cell is the navigation console.

> If something fails immediately, check the printed paths before assuming the hydrology has betrayed you.


In [ ]:
notebook_dir = Path.cwd()
home_dir = notebook_dir.parent

sim_name = "mfsim"
base_model_workspace = home_dir / "modflow_models" / "iisg_local_model"
age_model_workspace = home_dir / "modflow_models" / "groundwater_age_model"

exe_name = home_dir / "bin" / "mf6_latest_2026-04.exe"

county_shapefile = home_dir / "shapefiles" / "counties_5070.shp"

print("Base model workspace:", base_model_workspace)
print("Groundwater age model workspace:", age_model_workspace)
print("MODFLOW 6 executable:", exe_name)
print("County shapefile:", county_shapefile)


## 3. Load the base groundwater-flow model

The base model is the starting point for everything that follows. It contains the model grid, aquifer properties, boundary conditions, wells, streams, drains, and the groundwater-flow solution that the transport models build on.

We use FloPy's `MFSimulation.load()` to read the existing MODFLOW 6 simulation from disk. The model is not changed in this step. We are simply opening the model and checking that the groundwater-flow model object is available.

### What this gives us

- The model grid and layer structure
- The active and inactive cells
- Hydraulic properties and boundary packages
- Existing simulated heads and budget information
- A trusted starting point for the age and reverse-capture experiments

Think of this as opening the full map of the system before deciding where the mission should go next.


In [ ]:
base_sim = flopy.mf6.MFSimulation.load(
    sim_name=sim_name,
    sim_ws=base_model_workspace,
    write_headers=False,
    exe_name=exe_name,
)

base_model_name = list(base_sim.model_names)[0]
base_gwf = base_sim.get_model(base_model_name)

print("Loaded base model:", base_model_name)


## 4. Create the groundwater-age working copy

We do not modify the original base model directly. Instead, this cell writes a working copy of the simulation to a new groundwater-age workspace and then reloads that copy.

That may feel like extra ceremony, but it is a useful modeling habit. The base model remains clean, repeatable, and available for other scenarios. The age model gets its own workspace where we can add transport packages and output files without mixing them into the original simulation.

### 🌌 A Water Modeling Parable: The Stolen Shuttle

The Rebels do not fly the whole fleet straight into the shield gate without preparation. They use a specific mission vehicle, with a specific purpose, for a specific operation. This working copy is our shuttle: it carries the same core model information, but it gives us a safe place to add the groundwater-age machinery.


In [ ]:
base_sim.set_sim_path(age_model_workspace)
base_sim.write_simulation()

# Restore the base simulation path after writing the copy.
base_sim.set_sim_path(base_model_workspace)

age_sim = flopy.mf6.MFSimulation.load(
    sim_name=sim_name,
    sim_ws=age_model_workspace,
    write_headers=False,
    exe_name=exe_name,
)

age_model_name = list(age_sim.model_names)[0]
age_gwf = age_sim.get_model(age_model_name)

print("Loaded groundwater-age model:", age_model_name)
print("Age model workspace OK:", Path(age_gwf.model_ws) == age_model_workspace)


## 5. Initialize groundwater-age starting heads

Groundwater age is calculated by pairing a groundwater-flow model with a groundwater-transport model. Before we add the age transport model, we update the copied model's initial heads using the final heads from the base simulation.

Heads are simulated groundwater levels. They control hydraulic gradients, and hydraulic gradients control the direction and strength of groundwater movement. Starting from the base-model head field keeps the age calculation tied to the same flow system used elsewhere in the workshop.

### Why this matters

If the starting heads are inconsistent with the base model, the transport model can inherit a flow field that does not represent the intended aquifer condition. For a public walkthrough, this is like beginning the story on the wrong moon. The maps might still run, but they would be harder to trust.


In [ ]:
base_headobj = base_gwf.output.head()
base_head_times = base_headobj.get_times()

if len(base_head_times) == 0:
    raise RuntimeError(
        "No base-model head output was found. Run the base model first, then rerun this cell."
    )

base_final_time = base_head_times[-1]
base_final_heads = np.array(base_headobj.get_data(totim=base_final_time), dtype=float)

age_start_heads = base_final_heads.copy()
current_start_heads = np.array(age_gwf.ic.strt.array, dtype=float)
idomain = np.array(age_gwf.dis.idomain.array)

bad_heads = (
    ~np.isfinite(age_start_heads)
    | (age_start_heads > 1.0e20)
    | (age_start_heads < -1.0e20)
    | (idomain <= 0)
)

age_start_heads[bad_heads] = current_start_heads[bad_heads]
age_gwf.ic.strt.set_data(age_start_heads)

print(f"Updated age-model initial heads from base model heads at time {base_final_time}.")
print(f"Initial head range: {np.nanmin(age_start_heads):.2f} to {np.nanmax(age_start_heads):.2f}")


## 6. Configure the groundwater-age run

This cell defines the time discretization for the groundwater-age model. The age simulation is run long enough to let regional age patterns develop across the modeled aquifer system.

The important settings are:

| Setting | Meaning |
|---|---|
| `age_num_periods` | Number of MODFLOW stress periods used for the age run |
| `age_length_days` | Total simulated time for the age field |
| `age_time_steps` | Number of numerical time steps used inside the run |
| `age_timestep_mult` | Time-step growth factor |

The groundwater-age run is separate from the reverse-capture run later in the notebook. The age model is a regional context map. The reverse-capture model is a focused connection test around pumping wells.

### 🌌 A Water Modeling Parable: The Rancor Pit Has a Clock

In Jabba's palace, the rancor pit is terrifying because once someone falls in, time matters immediately. In an aquifer, time matters too, but on a much slower and less cinematic scale. Groundwater may take years, decades, or centuries to move through different parts of the system. This cell sets the clock so the model can reveal those longer patterns.


In [ ]:
# Keep this as the longer age-field run. The 25-year / 5-10-25 output
# request is handled in the reverse-capture GWT section below.

age_num_periods = 1
age_length_days = 365.25 * 500
age_time_steps = 25
age_timestep_mult = 1.05

age_time_dis = age_sim.get_package("tdis")
age_time_dis.nper = age_num_periods
age_time_dis.perioddata = [
    (age_length_days, age_time_steps, age_timestep_mult)
]

print(age_time_dis.perioddata.get_data())


## 7. Build the groundwater-age transport model

Here we add a groundwater-transport model to the groundwater-flow simulation. In this notebook, the transported quantity is **age**, not a contaminant plume.

The basic idea follows the direct groundwater-age approach described by Goode (1996): age can be simulated using an advection-dispersion-style transport equation, where water effectively accumulates age as it moves through the groundwater system. Source water is treated as young water, and the model tracks how that age field develops through the simulated flow system.

### What the setup means

| Model component | Interpretation |
|---|---|
| Initial age concentration = 0 | Water starts with no assigned age at the beginning of the age transport run |
| Porosity | Controls how transport storage relates to the aquifer material |
| Negative zero-order decay | The modeling trick that causes age to accumulate through time |
| Output control | Saves the final age field for mapping |

### 🌌 A Water Modeling Parable: The Force Has a Long Memory

The Force connects living things across space and time. Groundwater is not mystical, but it does carry memory. Water that recently entered the system and water that has traveled slowly through the aquifer for a long time can behave very differently. The age model helps us see that hidden memory as a map.


In [ ]:
age_gwt_name = f"{age_model_name}_gwt"

age_gwt = flopy.mf6.ModflowGwt(
    age_sim,
    modelname=age_gwt_name,
)

flopy.mf6.ModflowGwtdis(
    age_gwt,
    nlay=age_gwf.dis.nlay.array,
    nrow=age_gwf.dis.nrow.array,
    ncol=age_gwf.dis.ncol.array,
    delr=age_gwf.dis.delr.array,
    delc=age_gwf.dis.delc.array,
    top=age_gwf.dis.top.array,
    botm=age_gwf.dis.botm.array,
    idomain=age_gwf.dis.idomain.array,
    xorigin=age_gwf.modelgrid.xoffset,
    yorigin=age_gwf.modelgrid.yoffset,
    angrot=age_gwf.modelgrid.angrot,
)

print("\nGWF modelgrid extent:")
print(age_gwf.modelgrid.extent)

print("\nGWT modelgrid extent:")
print(age_gwt.modelgrid.extent)

# Initial age concentration = 0 everywhere.
flopy.mf6.ModflowGwtic(
    age_gwt,
    strt=0.0,
)

flopy.mf6.ModflowGwtadv(
    age_gwt,
    scheme="UPSTREAM",
)

# Porosity: use STO sy if available; otherwise use a constant fallback.
if age_gwf.get_package("sto") is not None and hasattr(age_gwf.sto, "sy"):
    porosity = np.array(age_gwf.sto.sy.array, dtype=float)
    porosity = np.where((porosity > 0) & np.isfinite(porosity), porosity, 0.15)
else:
    porosity = np.full(
        (
            age_gwf.modelgrid.nlay,
            age_gwf.modelgrid.nrow,
            age_gwf.modelgrid.ncol,
        ),
        0.15,
        dtype=float,
    )

# Groundwater-age trick:
# zero-order decay with negative decay produces age concentration over time.
age_decay_rate = -1.0  # age units per model day

flopy.mf6.ModflowGwtmst(
    age_gwt,
    porosity=porosity,
    zero_order_decay=True,
    decay=age_decay_rate,
    save_flows=False,
)

# Treat source water as zero-age water.
flopy.mf6.ModflowGwtssm(age_gwt)

# Save only the final total-age concentration field.
flopy.mf6.ModflowGwtoc(
    age_gwt,
    concentration_filerecord=f"{age_gwt_name}.ucn",
    saverecord=[
        ("CONCENTRATION", "LAST"),
    ],
)

# Add MVT if the GWF model has movers.
has_mvr = any("MVR" in package_name.upper() for package_name in age_gwf.get_package_list())

if has_mvr:
    flopy.mf6.ModflowGwtmvt(
        age_gwt,
        save_flows=True,
    )

age_ims = flopy.mf6.ModflowIms(
    age_sim,
    filename=f"{age_gwt.name}.ims",
    print_option="summary",
    complexity="simple",
    outer_maximum=100,
    inner_maximum=50,
    outer_dvclose=1.0e-2,
    inner_dvclose=1.0e-2,
    linear_acceleration="bicgstab",
)

age_sim.register_ims_package(
    age_ims,
    model_list=[f"{age_gwt.name}"],
)

flopy.mf6.ModflowGwfgwt(
    age_sim,
    exgtype="GWF6-GWT6",
    exgmnamea=age_gwf.name,
    exgmnameb=age_gwt.name,
)

print(f"Added GWT groundwater-age model: {age_gwt_name}")


## 8. Run the groundwater-age model

This cell writes the updated simulation files and runs MODFLOW 6. A successful run produces the transport output used in the total groundwater-age map.

Watch the console output for **normal termination**. That message means MODFLOW completed the run successfully. If the run fails, the listing file and console messages are usually the first places to look for convergence issues, missing files, or package-input problems.

### 🌌 A Water Modeling Parable: Launching the Fleet

Once the briefing is over, the fleet has to launch. This is the launch cell. Up to this point, we have been preparing files and adding model instructions. Now MODFLOW actually performs the calculation.


In [ ]:
age_sim.write_simulation()

success, buff = age_sim.run_simulation()

if success:
    print("\n-- Groundwater age simulation completed successfully --\n")
else:
    raise RuntimeError("Groundwater age simulation failed. Check the listing file.")


## 9. Plot total groundwater age

The total-age map shows the final simulated groundwater age for the selected model layer. The plot is cropped to the active model domain so the figure focuses on the aquifer system rather than the inactive rectangular border around the grid.

Read this as a **pattern map**. It is most useful for comparing areas across the model domain:

| Map pattern | General interpretation |
|---|---|
| Younger groundwater | Faster connection to recharge, boundaries, or active flow paths |
| Older groundwater | Longer flow paths, slower movement, or more isolated parts of the aquifer |
| Sharp changes | Possible transitions between hydrogeologic units, boundaries, or flow regimes |

### 🌌 A Water Modeling Parable: Endor Is Not One Forest

From orbit, Endor looks like one continuous green moon. On the ground, it is a complicated landscape of villages, paths, ridges, traps, and shield-generator infrastructure. The aquifer is similar. A regional model domain may look continuous from above, but the age map reveals internal structure: some parts are strongly connected and active, while others move more slowly or behave differently.

Older groundwater is not automatically bad, and younger groundwater is not automatically good. The point is to understand how the system is organized before making decisions about pumping, monitoring, or protection.


In [ ]:
# This keeps the whole active model domain visible while trimming the large
# inactive/blank grid area around it.

age_plot_layer = 8  # zero-based; 8 = model layer 9
plot_age_in_years = True
age_domain_pad_cells = 5

ucn = age_gwt.output.concentration()
times = ucn.get_times()

if len(times) == 0:
    raise RuntimeError("No groundwater-age concentration output was found.")

final_time = times[-1]
age_conc = np.array(ucn.get_data(totim=final_time), dtype=float)

age_map = age_conc[age_plot_layer].copy()
age_map[age_map > 1.0e20] = np.nan
age_map[age_map < -1.0e20] = np.nan

idomain = np.array(age_gwf.dis.idomain.array)
age_map = np.where(idomain[age_plot_layer] > 0, age_map, np.nan)

if plot_age_in_years:
    age_map = age_map / 365.25
    age_units = "years"
else:
    age_units = "days"

active_any_layer = np.any(idomain > 0, axis=0)
active_rows, active_cols = np.where(active_any_layer)

if active_rows.size == 0 or active_cols.size == 0:
    raise RuntimeError("No active cells found in idomain.")

row_start = max(int(active_rows.min()) - age_domain_pad_cells, 0)
row_end = min(int(active_rows.max()) + age_domain_pad_cells + 1, age_map.shape[0])
col_start = max(int(active_cols.min()) - age_domain_pad_cells, 0)
col_end = min(int(active_cols.max()) + age_domain_pad_cells + 1, age_map.shape[1])

age_map_zoom = age_map[row_start:row_end, col_start:col_end]

xcenters = age_gwf.modelgrid.xcellcenters
ycenters = age_gwf.modelgrid.ycellcenters

x_zoom = xcenters[row_start:row_end, col_start:col_end]
y_zoom = ycenters[row_start:row_end, col_start:col_end]

dx = np.nanmedian(np.abs(np.diff(xcenters, axis=1)))
dy = np.nanmedian(np.abs(np.diff(ycenters, axis=0)))

if not np.isfinite(dx):
    dx = 0.0
if not np.isfinite(dy):
    dy = 0.0

x_min = np.nanmin(x_zoom) - dx / 2
x_max = np.nanmax(x_zoom) + dx / 2
y_min = np.nanmin(y_zoom) - dy / 2
y_max = np.nanmax(y_zoom) + dy / 2

valid_age = age_map_zoom[np.isfinite(age_map_zoom)]

if valid_age.size > 0:
    vmin = 0.0
    vmax = np.nanpercentile(valid_age, 95)
else:
    vmin = 0.0
    vmax = 1.0

counties = gpd.read_file(county_shapefile)

try:
    model_crs = age_gwf.modelgrid.crs
except Exception:
    model_crs = None

if model_crs is not None and counties.crs is not None and counties.crs != model_crs:
    counties = counties.to_crs(model_crs)

counties_plot = counties.cx[x_min:x_max, y_min:y_max]

with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=(10, 12))
    ax.set_aspect("equal")

    im = ax.imshow(
        np.ma.masked_invalid(age_map_zoom),
        origin="upper",
        extent=[x_min, x_max, y_min, y_max],
        vmin=vmin,
        vmax=100, #vmax,
    )

    if len(counties_plot) > 0:
        counties_plot.boundary.plot(ax=ax, linewidth=0.8)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_title(
        f"Total groundwater age, layer {age_plot_layer + 1}\n"
        f"final simulation time = {final_time / 365.25:.1f} years"
    )
    ax.set_xlabel("x")
    ax.set_ylabel("y")

    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label(f"groundwater age ({age_units})")

    plt.show()

print("Plotted total groundwater age for active-domain extent.")
print(f"Rows plotted: {row_start} to {row_end - 1}")
print(f"Cols plotted: {col_start} to {col_end - 1}")


## 10. Reverse-capture setup

The reverse-capture analysis asks a different question from the age map:

> If groundwater reaches these pumping wells, where does the model suggest that water may have come from during the selected lookback period?

To answer that, the notebook runs a reverse-transport experiment. A unit tracer signal is assigned at pumping wells, the saved flow information is reversed for transport, and the GWT model tracks that signal backward through the modeled groundwater connections.

For this presentation notebook, we only need three capture-zone snapshots: **5 years**, **10 years**, and **25 years**. Instead of preparing 25 one-year records, the reverse workflow below uses three model periods that end exactly at those discussion times. That keeps the run lighter while still producing the three maps we want to talk through.

### 🌌 A Water Modeling Parable: Jabba's Palace and the Rancor Pit

A pumping well is not a monster, but it does create a destination in the groundwater-flow system. In reverse capture, we start at that destination and ask which pathways could lead back to it. The pumping wells are the rancor pit in this analogy: not because the water is in danger, but because the analysis is organized around what eventually reaches that location.

The threshold matters. A very tiny reverse-tracer concentration can spread broadly through the model because of numerical dispersion, grid resolution, and solver behavior. That is where **Salacious B. Crumb** enters the notebook: a faint giggle from the corner of the palace does not mean the whole galaxy is in the rancor pit. For the main map, we use a stronger signal threshold (`C > 0.05`) so the footprint represents cells with at least 5 percent of the unit reverse-tracer signal.


In [ ]:
reverse_forward_workspace = Path(home_dir / "modflow_models" / "reverse_capture_forward_flow")
reverse_gwt_workspace = Path(home_dir / "modflow_models" / "reverse_capture_gwt_test")

# Capture-zone snapshots to prepare and plot.
# The model will use three periods ending at 5, 10, and 25 years.
reverse_output_years = [5.0, 10.0, 25.0]
reverse_period_years = np.diff([0.0] + reverse_output_years).tolist()
reverse_perioddata = [(years * 365.25, 1, 1.0) for years in reverse_period_years]
reverse_nper = len(reverse_perioddata)
reverse_lookback_years = reverse_output_years[-1]
reverse_lookback_days = reverse_lookback_years * 365.25

# Unit signal used for the reverse-tracer source at the pumping wells.
# Capture-zone maps show fractions of this signal, so 1.0 makes the
# concentration field easy to read as relative connection strength.
reverse_source_concentration = 1.0

# Main capture-zone threshold.
# A cell is mapped when the reverse-tracer concentration exceeds 5%
# of the unit signal assigned at the pumping well boundary.
capture_concentration_threshold = 0.05

# Auto-pick the first WEL package and all extracting wells with q < 0.
# Override these only if you want a narrower well set.
reverse_wel_package_name = None
reverse_wel_bndnos = None

# Local capture-zone map window.
# These are model coordinates; adjust here for a different local view.
local_capture_x_min = 680000.0
local_capture_x_max = 700000.0
local_capture_y_min = 2.08e6
local_capture_y_max = 2.10e6

active_domain_pad_cells = 5

run_reverse_forward_flow = True
run_reverse_gwt = True

print("Reverse forward-flow workspace:", reverse_forward_workspace)
print("Reverse GWT workspace:", reverse_gwt_workspace)
print("Reverse capture-zone years:", reverse_output_years)
print("Reverse period lengths, in years:", reverse_period_years)
print("Reverse perioddata:", reverse_perioddata)


## 11. Reverse-capture helper functions

The reverse-capture workflow repeats a few tasks: finding pumping wells, writing the source-concentration file, selecting map windows, and turning concentration arrays into plan-view capture-zone masks.

The helper functions keep those details in one place so the rest of the notebook reads like a modeling sequence rather than a pile of repeated code.

### 🌌 A Water Modeling Parable: Ewok Engineering

The Ewoks do not win because they have one giant machine. They win because many small, well-placed pieces work together: ropes, logs, traps, timing, and local knowledge. Helper functions are the same idea in code. Each one does a small job clearly, and together they make the larger workflow easier to follow.


In [ ]:
def first_package(model, package_type):
    """Return the first package of a given MODFLOW 6 package type."""
    package_type = package_type.lower()
    for pkg in model.packagelist:
        if str(getattr(pkg, "package_type", "")).lower() == package_type:
            return pkg
    raise RuntimeError(f"No {package_type.upper()} package found.")


def package_name(pkg):
    """Return the FloPy package name as a plain string."""
    name = getattr(pkg, "package_name", None)
    if name is None:
        name = getattr(pkg, "pname", None)
    if hasattr(name, "array"):
        name = name.array
    if isinstance(name, (list, tuple, np.ndarray)):
        name = name[0]
    return str(name)


def final_heads_from_base(source_gwf, target_gwf):
    """Use the final base-model heads as starting heads for a copied model."""
    hds = source_gwf.output.head()
    final_time = hds.get_times()[-1]
    heads = np.array(hds.get_data(totim=final_time), dtype=float)

    idomain = np.array(target_gwf.dis.idomain.array)
    current = np.array(target_gwf.ic.strt.array, dtype=float)

    bad = (~np.isfinite(heads)) | (np.abs(heads) > 1.0e20) | (idomain <= 0)
    heads[bad] = current[bad]
    return heads, final_time


def choose_pumping_wells(gwf, kper=0):
    """Use all extracting wells in the first WEL package as reverse-tracer sources."""
    wel = first_package(gwf, "wel")
    data = wel.stress_period_data.get_data(key=kper)
    q = np.asarray(data["q"], dtype=float)

    bndnos = list(np.where(q < 0.0)[0] + 1)  # SPC6 boundary numbers are 1-based.
    maxbound = int(np.asarray(wel.maxbound.array).item())

    return wel, package_name(wel), bndnos, maxbound


def write_spc6_file(spc_path, bndnos, concentration, maxbound):
    """Write the SPC6 file that assigns reverse-tracer concentration to selected wells."""
    spc_path = Path(spc_path)
    spc_path.parent.mkdir(parents=True, exist_ok=True)

    with open(spc_path, "w") as f:
        f.write("BEGIN OPTIONS\n  PRINT_INPUT\nEND OPTIONS\n\n")
        f.write(f"BEGIN DIMENSIONS\n  MAXBOUND {maxbound}\nEND DIMENSIONS\n\n")
        f.write("BEGIN PERIOD 1\n")
        for bndno in bndnos:
            f.write(f"  {int(bndno)} CONCENTRATION {float(concentration):.16g}\n")
        f.write("END PERIOD\n")

    return spc_path


def turn_on_flow_output(gwf):
    """Save the flow terms needed by the reverse transport model."""
    for pkg in gwf.packagelist:
        if hasattr(pkg, "save_flows"):
            try:
                pkg.save_flows.set_data(True)
            except Exception:
                pass

    gwf.npf.save_flows.set_data(True)
    gwf.npf.save_specific_discharge.set_data(True)
    gwf.npf.save_saturation.set_data(True)


## 12. Create the forward-flow files that will be reversed

The reverse-transport model needs saved forward-flow output before that output can be reversed. This cell creates a short supporting groundwater-flow run and saves the head and budget files needed by the reverse-GWT calculation.

This is **not** a new management scenario. It is a file-preparation step. The forward run records the flow information that will later be turned around for the reverse-capture experiment.

### Why this step exists

GWT needs flow information to move the tracer. For a reverse-capture analysis, we first save the normal forward-flow information, then reverse the relevant binary files. Without this supporting run, the reverse-transport model would not have the flow budget information it needs.

### Starting close to the answer

This copied flow model does **not** start from a blank guess. It uses the final heads from the base-model solution as its starting heads. That usually helps the solver converge faster because the supporting run begins near a reasonable hydraulic surface instead of wandering in from a cold start.

To keep the presentation version practical, this supporting run now saves only the three times we need: **5**, **10**, and **25** years. Those are the three snapshots used for the capture-zone maps. Fewer saved records means less file preparation, less transport bookkeeping, and a notebook that is easier to rerun during map-making.

### 🌌 A Water Modeling Parable: You Cannot Reverse a Speeder-Bike Chase You Did Not Record

Before we can show the same pathway backward, we need the original pathway. This cell records the forward movement through the model system at the moments we plan to discuss, so the next steps can turn those records around.


In [ ]:
reverse_forward_workspace.mkdir(parents=True, exist_ok=True)

# Write a copy of the base model to the reverse-forward workspace.
base_sim.set_sim_path(reverse_forward_workspace)
base_sim.write_simulation()
base_sim.set_sim_path(base_model_workspace)

reverse_forward_sim = flopy.mf6.MFSimulation.load(
    sim_name=sim_name,
    sim_ws=reverse_forward_workspace,
    write_headers=False,
    exe_name=exe_name,
)

reverse_forward_model_name = list(reverse_forward_sim.model_names)[0]
reverse_forward_gwf = reverse_forward_sim.get_model(reverse_forward_model_name)

reverse_start_heads, reverse_base_final_time = final_heads_from_base(base_gwf, reverse_forward_gwf)
reverse_forward_gwf.ic.strt.set_data(reverse_start_heads)

reverse_forward_sim.tdis.nper = reverse_nper
reverse_forward_sim.tdis.perioddata = reverse_perioddata

if reverse_forward_gwf.get_package("sto") is not None:
    reverse_forward_gwf.sto.steady_state.set_data({iper: True for iper in range(reverse_nper)})
    reverse_forward_gwf.sto.transient.set_data({iper: False for iper in range(reverse_nper)})

reverse_forward_head_file = reverse_forward_workspace / f"{reverse_forward_gwf.name}.hds"
reverse_forward_budget_file = reverse_forward_workspace / f"{reverse_forward_gwf.name}.cbc"

reverse_forward_gwf.oc.head_filerecord.set_data(reverse_forward_head_file.name)
reverse_forward_gwf.oc.budget_filerecord.set_data(reverse_forward_budget_file.name)
reverse_forward_gwf.oc.saverecord.set_data(
    {iper: [("HEAD", "LAST"), ("BUDGET", "LAST")] for iper in range(reverse_nper)}
)
reverse_forward_gwf.oc.printrecord.set_data(
    {iper: [("BUDGET", "LAST")] for iper in range(reverse_nper)}
)

turn_on_flow_output(reverse_forward_gwf)

reverse_forward_sim.write_simulation()

if run_reverse_forward_flow:
    success, buff = reverse_forward_sim.run_simulation(silent=False, report=True)
    if not success:
        raise RuntimeError("Forward-flow run for reverse capture failed.")

print("Forward head file:", reverse_forward_head_file)
print("Forward budget file:", reverse_forward_budget_file)


## 13. Select pumping wells and write the reverse-source file

The model's WEL package identifies pumping boundaries. This cell selects the extracting wells and assigns them a unit concentration in the SPC6 source file used by the reverse-GWT model.

That unit concentration is a modeling signal, not a real chemical. It is a beacon placed at the pumping wells so the transport model can trace the connected pathways backward through the reversed flow field.

### What to remember

- The selected wells define the focus of the capture-zone analysis
- The unit source makes the reverse-tracer signal easy to interpret as relative connection strength
- The result is not a simulated contaminant plume; it is a modeled connection footprint

### 🌌 A Water Modeling Parable: The Shield Generator Beacon

On Endor, the shield generator is the strategic target because it controls the larger battle. In this notebook, the pumping wells are the strategic target because they define the destination we care about. The unit tracer marks that target so the model can ask: which parts of the aquifer are connected to it through time?


In [ ]:
reverse_wel_pkg, reverse_wel_pname, reverse_wel_bndnos, reverse_wel_maxbound = choose_pumping_wells(
    reverse_forward_gwf,
    kper=0,
)

reverse_spc_file = reverse_gwt_workspace / f"{reverse_wel_pname}.spc6"
write_spc6_file(
    spc_path=reverse_spc_file,
    bndnos=reverse_wel_bndnos,
    concentration=reverse_source_concentration,
    maxbound=reverse_wel_maxbound,
)

print("Reverse-source WEL package:", reverse_wel_pname)
print("Number of pumping wells used as reverse sources:", len(reverse_wel_bndnos))
print("SPC6 file:", reverse_spc_file)


## 14. Reverse the saved head and budget files

The forward-flow run produced the binary head and cell-by-cell budget files needed for transport. This cell uses FloPy's binary-file utilities to create reversed versions for the backward transport calculation.

The budget file is the key piece for the capture-zone workflow because it contains the intercell flow information used by GWT. Reversing that flow information lets the tracer signal move backward through the modeled connections instead of forward with groundwater flow.

The heads themselves are not expected to become a mirror image. Heads describe the hydraulic surface from the flow solution. The reverse-capture experiment changes the direction used by the transport calculation, not the underlying idea of what the head field represents.

### 🌌 A Water Modeling Parable: Turning the Speeder Bike Around

The forest does not change when a scout trooper turns around. The trees, slopes, and paths are still there. What changes is the direction of travel through that landscape. That is the idea here: the model domain remains the same, but the transport direction is reversed.


In [ ]:
reverse_head_file = reverse_gwt_workspace / f"{reverse_forward_gwf.name}_reverse.hds"
reverse_budget_file = reverse_gwt_workspace / f"{reverse_forward_gwf.name}_reverse.cbc"

reverse_gwt_workspace.mkdir(parents=True, exist_ok=True)

headobj = bf.HeadFile(str(reverse_forward_head_file), tdis=reverse_forward_sim.tdis)
headobj.reverse(filename=str(reverse_head_file))

cbb = bf.CellBudgetFile(str(reverse_forward_budget_file), tdis=reverse_forward_sim.tdis)
cbb.reverse(filename=str(reverse_budget_file))

print("Reversed head file:", reverse_head_file)
print("Reversed budget file:", reverse_budget_file)


## 15. Build and run the reverse transport model

This cell builds a GWT-only model connected to the reversed flow information. The model starts with a unit tracer signal at the selected pumping wells and moves that signal backward through the modeled flow field.

The presentation version uses three transport periods: one ending at 5 years, one ending at 10 years, and one ending at 25 years. The output times therefore match the three capture-zone maps directly.

This is faster and easier to discuss than running every single year. The tradeoff is that the transport calculation is coarser in time. For a steady-flow screening map, that is usually a reasonable presentation choice. If the edge of the capture zone becomes the focus of a regulatory or design decision, a finer time-step sensitivity check is still worth doing.

### How to read the reverse-tracer concentration

The concentration is best interpreted as **relative connection strength**. Higher values indicate a stronger modeled connection to the selected pumping wells during the selected time window. Lower values indicate weaker or more diffuse connection.

This is why the concentration threshold matters. The transport solution fades outward rather than ending at a perfectly sharp line. A capture-zone map is therefore a defined planning footprint, not a magic underground wall.

### 🌌 A Water Modeling Parable: The Second Death Star Is a System, Not a Point

The Death Star II may look like one target, but the battle depends on the shield generator, the fleet, the forest moon, and the timing of many moving pieces. A pumping well is also not just a point. It is connected to a larger flow system. Reverse GWT helps us map those connections in a way people can discuss.


In [ ]:
reverse_gwt_name = "rvcap_gwt"  # MF6 MODELNAME must be <= 16 characters

reverse_gwt_sim = flopy.mf6.MFSimulation(
    sim_name="reverse_capture_gwt_sim",
    sim_ws=reverse_gwt_workspace,
    exe_name=exe_name,
)

flopy.mf6.ModflowTdis(
    reverse_gwt_sim,
    time_units="DAYS",
    nper=reverse_nper,
    perioddata=reverse_perioddata,
)

reverse_gwt = flopy.mf6.ModflowGwt(
    reverse_gwt_sim,
    modelname=reverse_gwt_name,
    save_flows=True,
)

flopy.mf6.ModflowGwtdis(
    reverse_gwt,
    nlay=reverse_forward_gwf.dis.nlay.array,
    nrow=reverse_forward_gwf.dis.nrow.array,
    ncol=reverse_forward_gwf.dis.ncol.array,
    delr=reverse_forward_gwf.dis.delr.array,
    delc=reverse_forward_gwf.dis.delc.array,
    top=reverse_forward_gwf.dis.top.array,
    botm=reverse_forward_gwf.dis.botm.array,
    idomain=reverse_forward_gwf.dis.idomain.array,
    xorigin=reverse_forward_gwf.modelgrid.xoffset,
    yorigin=reverse_forward_gwf.modelgrid.yoffset,
    angrot=reverse_forward_gwf.modelgrid.angrot,
)

flopy.mf6.ModflowGwtic(
    reverse_gwt,
    strt=0.0,
)

flopy.mf6.ModflowGwtadv(
    reverse_gwt,
    scheme="UPSTREAM",
)

if reverse_forward_gwf.get_package("sto") is not None and hasattr(reverse_forward_gwf.sto, "sy"):
    reverse_porosity = np.array(reverse_forward_gwf.sto.sy.array, dtype=float)
    reverse_porosity = np.where((reverse_porosity > 0) & np.isfinite(reverse_porosity), reverse_porosity, 0.15)
else:
    reverse_porosity = np.full(
        (
            reverse_forward_gwf.modelgrid.nlay,
            reverse_forward_gwf.modelgrid.nrow,
            reverse_forward_gwf.modelgrid.ncol,
        ),
        0.15,
        dtype=float,
    )

flopy.mf6.ModflowGwtmst(
    reverse_gwt,
    porosity=reverse_porosity,
    save_flows=True,
)

flopy.mf6.ModflowGwtfmi(
    reverse_gwt,
    save_flows=True,
    packagedata=[
        ("GWFHEAD", str(reverse_head_file)),
        ("GWFBUDGET", str(reverse_budget_file)),
    ],
)

flopy.mf6.ModflowGwtssm(
    reverse_gwt,
    print_flows=True,
    save_flows=True,
    fileinput=[
        (reverse_wel_pname, str(reverse_spc_file)),
    ],
)

# Save concentration at the end of each period: 5, 10, and 25 years.
reverse_gwt_oc_records = {
    iper: [("CONCENTRATION", "LAST")]
    for iper in range(reverse_nper)
}

flopy.mf6.ModflowGwtoc(
    reverse_gwt,
    concentration_filerecord=f"{reverse_gwt_name}.ucn",
    saverecord=reverse_gwt_oc_records,
    printrecord=reverse_gwt_oc_records,
)

reverse_gwt_ims = flopy.mf6.ModflowIms(
    reverse_gwt_sim,
    filename=f"{reverse_gwt_name}.ims",
    print_option="summary",
    complexity="simple",
    outer_maximum=100,
    inner_maximum=100,
    outer_dvclose=1.0e-6,
    inner_dvclose=1.0e-6,
    linear_acceleration="bicgstab",
)

reverse_gwt_sim.register_ims_package(
    reverse_gwt_ims,
    model_list=[reverse_gwt.name],
)

reverse_gwt_sim.write_simulation()

if run_reverse_gwt:
    success, buff = reverse_gwt_sim.run_simulation(silent=False, report=True)
    if not success:
        raise RuntimeError("Reverse GWT run failed. Check the reverse GWT listing file.")
    print("\n-- Reverse GWT simulation completed --\n")
else:
    print("run_reverse_gwt = False, so the reverse GWT simulation was written but not executed.")

reverse_ucn = reverse_gwt.output.concentration()
print("Saved reverse-GWT concentration output times, in years:")
print([t / 365.25 for t in reverse_ucn.get_times()])


## 16. Plot reverse-capture zones

The capture-zone maps convert the reverse-tracer concentration field into nested travel-time footprints. A row and column appears on the plan-view map when **any active model layer** beneath it exceeds the selected concentration threshold.

| Zone | Interpretation |
|---|---|
| 5-year zone | Stronger modeled connection within 5 years |
| 10-year zone | Additional area connected within 10 years |
| 25-year zone | Broader footprint connected within 25 years |

The active-domain map shows the full modeled footprint. The local map zooms in so the result can be discussed at a scale closer to familiar roads, boundaries, wells, and community questions.

### 🌌 A Water Modeling Parable: Ewok Trails Through the Forest

The Ewoks know that the forest is not just open space. It is a network of trails, traps, overlooks, and shortcuts. The capture-zone map is similar: it shows where the model sees meaningful pathways connected to the pumping wells over time.

Because the map collapses all layers into one plan view, a colored surface cell does not necessarily mean the shallowest layer is the source. It means at least one active aquifer layer below that row and column crossed the threshold. That makes the map useful for regional screening and discussion, while still reminding us that the vertical structure of the aquifer matters.

### What to look for

- Do the 5-, 10-, and 25-year zones expand in a way that makes hydrogeologic sense?
- Are the strongest patterns continuous, patchy, or tied to particular model features?
- Which edges look stable, and which edges would benefit from sensitivity testing or local review?


In [ ]:
def model_counties(gwf):
    counties = gpd.read_file(county_shapefile)
    model_crs = gwf.modelgrid.crs
    if model_crs is not None and counties.crs is not None and counties.crs != model_crs:
        counties = counties.to_crs(model_crs)
    return counties


def active_window(idomain, pad_cells=5):
    active = np.any(idomain > 0, axis=0)
    rows, cols = np.where(active)
    return (
        max(rows.min() - pad_cells, 0),
        min(rows.max() + pad_cells + 1, active.shape[0]),
        max(cols.min() - pad_cells, 0),
        min(cols.max() + pad_cells + 1, active.shape[1]),
    )


def xy_window(gwf, idomain, x_min, x_max, y_min, y_max):
    x = gwf.modelgrid.xcellcenters
    y = gwf.modelgrid.ycellcenters
    active = np.any(idomain > 0, axis=0)
    inside = active & (x >= x_min) & (x <= x_max) & (y >= y_min) & (y <= y_max)
    rows, cols = np.where(inside)
    return rows.min(), rows.max() + 1, cols.min(), cols.max() + 1


def window_extent(gwf, row0, row1, col0, col1):
    x = gwf.modelgrid.xcellcenters
    y = gwf.modelgrid.ycellcenters
    dx = np.nanmedian(np.abs(np.diff(x, axis=1)))
    dy = np.nanmedian(np.abs(np.diff(y, axis=0)))
    return (
        np.nanmin(x[row0:row1, col0:col1]) - dx / 2,
        np.nanmax(x[row0:row1, col0:col1]) + dx / 2,
        np.nanmin(y[row0:row1, col0:col1]) - dy / 2,
        np.nanmax(y[row0:row1, col0:col1]) + dy / 2,
    )


def concentration_at_year(ucn, year):
    times = np.array(ucn.get_times(), dtype=float)
    target = year * 365.25
    t = times[np.argmin(np.abs(times - target))]
    return np.array(ucn.get_data(totim=t), dtype=float), t


def capture_masks(reverse_gwt, gwf, years, threshold):
    ucn = reverse_gwt.output.concentration()
    idomain = np.array(gwf.dis.idomain.array)
    masks = {}
    times = {}

    for year in years:
        conc, t = concentration_at_year(ucn, year)
        masks[year] = np.any((idomain > 0) & np.isfinite(conc) & (conc > threshold), axis=0)
        times[year] = t

    return masks, times


def plot_capture_zones(mode="active_domain", title="Reverse-GWT capture zones"):
    idomain = np.array(reverse_forward_gwf.dis.idomain.array)
    masks, times = capture_masks(
        reverse_gwt,
        reverse_forward_gwf,
        reverse_output_years,
        capture_concentration_threshold,
    )

    if mode == "active_domain":
        row0, row1, col0, col1 = active_window(idomain, active_domain_pad_cells)
    else:
        row0, row1, col0, col1 = xy_window(
            reverse_forward_gwf,
            idomain,
            local_capture_x_min,
            local_capture_x_max,
            local_capture_y_min,
            local_capture_y_max,
        )

    xmin, xmax, ymin, ymax = window_extent(reverse_forward_gwf, row0, row1, col0, col1)
    counties = model_counties(reverse_forward_gwf).cx[xmin:xmax, ymin:ymax]

    colors = {5.0: "tab:red", 10.0: "tab:orange", 25.0: "tab:blue"}
    fig, ax = plt.subplots(figsize=(11, 10))
    ax.set_aspect("equal")

    active = np.any(idomain > 0, axis=0)[row0:row1, col0:col1]
    ax.imshow(
        np.where(active, 1.0, np.nan),
        origin="upper",
        extent=[xmin, xmax, ymin, ymax],
        cmap=ListedColormap(["lightgray"]),
        alpha=0.25,
    )

    # Draw the largest time window first, then stack shorter travel times on top.
    for year in sorted(reverse_output_years, reverse=True):
        zone = masks[year][row0:row1, col0:col1]
        ax.imshow(
            np.where(zone, 1.0, np.nan),
            origin="upper",
            extent=[xmin, xmax, ymin, ymax],
            cmap=ListedColormap([colors[float(year)]]),
            alpha=0.65,
        )

    if len(counties) > 0:
        counties.boundary.plot(ax=ax, linewidth=0.8)

    handles = [
        Patch(facecolor=colors[float(year)], alpha=0.65, label=f"{year:g}-year zone")
        for year in sorted(reverse_output_years)
    ]
    ax.legend(handles=handles, loc="upper right")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"{title}\nthreshold C > {capture_concentration_threshold:g}")
    plt.show()

    for year in sorted(reverse_output_years):
        print(f"{year:g}-year cells:", int(masks[year].sum()), f"(output = {times[year] / 365.25:.1f} years)")


plot_capture_zones(mode="active_domain", title="Capture zones, active-domain overview")
plot_capture_zones(mode="local", title="Capture zones, local map")


## 17. Check the flow reversal

The reverse-capture run depends on one important mechanical step: the transport model must use flow directions that are turned around from the forward groundwater-flow solution.

This figure checks that idea in two ways. The left panel shows the forward head field with forward flow arrows. The right panel shows the same head context with the reversed transport-direction arrows. The printed budget check compares the `FLOW-JA-FACE` terms in the forward and reversed budget files; values near `-1` indicate that the intercell budget terms used by GWT have been sign-reversed.

### Why the head colors may look similar

The head surface is not expected to flip. Heads still show the modeled water-level pattern. What changes for reverse capture is the direction used by the tracer signal. In other words, the map can have similar head colors in both panels while the arrows point in opposite directions.

### 🌌 A Water Modeling Parable: Same Forest, Opposite Mission

Endor is still Endor whether the characters are sneaking toward the shield bunker or running away from it. The landscape does not reverse. The mission direction does. This check is our way of confirming that the reverse-capture model is using the same hydraulic setting but sending the tracer signal backward through it.

This is not a capture-zone result. It is a quality-control and explanation figure. It helps confirm that the reverse-capture maps are built from a reversed transport direction rather than from a forward-flow map wearing a fake mustache.


In [ ]:
flow_check_layer = age_plot_layer
flow_check_map_mode = "active_domain"  # "active_domain" or "local"
flow_vector_stride = 12 if flow_check_map_mode == "active_domain" else 4
flow_head_contour_count = 10


def read_heads(head_file):
    hds = bf.HeadFile(str(head_file), tdis=reverse_forward_sim.tdis)
    time = hds.get_times()[-1]
    heads = np.array(hds.get_data(totim=time), dtype=float)
    return heads, time


def read_spdis(budget_file):
    cbb = bf.CellBudgetFile(str(budget_file), tdis=reverse_forward_sim.tdis)
    time = cbb.get_times()[-1]
    spdis = cbb.get_data(text="DATA-SPDIS", totim=time)[0]
    qx, qy, qz = get_specific_discharge(spdis, reverse_forward_gwf)
    return np.array(qx), np.array(qy), time


def flowja_values(budget_file):
    cbb = bf.CellBudgetFile(str(budget_file), tdis=reverse_forward_sim.tdis)
    time = cbb.get_times()[-1]
    return np.asarray(cbb.get_data(text="FLOW-JA-FACE", totim=time)[0], dtype=float)


forward_heads, forward_head_time = read_heads(reverse_forward_head_file)
reversed_heads, reversed_head_time = read_heads(reverse_head_file)
forward_qx, forward_qy, forward_budget_time = read_spdis(reverse_forward_budget_file)

# DATA-SPDIS in a reversed budget file can still look like the original vector field.
# The GWT transport calculation uses the reversed cell-by-cell budget flows.
# For this visual check, plot the reverse transport direction directly.
reversed_qx = -forward_qx
reversed_qy = -forward_qy

idomain = np.array(reverse_forward_gwf.dis.idomain.array)

if flow_check_map_mode == "active_domain":
    row0, row1, col0, col1 = active_window(idomain, active_domain_pad_cells)
else:
    row0, row1, col0, col1 = xy_window(
        reverse_forward_gwf,
        idomain,
        local_capture_x_min,
        local_capture_x_max,
        local_capture_y_min,
        local_capture_y_max,
    )

xmin, xmax, ymin, ymax = window_extent(reverse_forward_gwf, row0, row1, col0, col1)
counties = model_counties(reverse_forward_gwf).cx[xmin:xmax, ymin:ymax]

x = reverse_forward_gwf.modelgrid.xcellcenters[row0:row1, col0:col1]
y = reverse_forward_gwf.modelgrid.ycellcenters[row0:row1, col0:col1]
active = idomain[flow_check_layer, row0:row1, col0:col1] > 0

pairs = [
    ("Forward heads and flow", forward_heads, forward_qx, forward_qy),
    ("Forward heads with reversed transport direction", reversed_heads, reversed_qx, reversed_qy),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 8), constrained_layout=True)

for ax, (title, heads, qx, qy) in zip(axes, pairs):
    h = np.where(active, heads[flow_check_layer, row0:row1, col0:col1], np.nan)
    qx_layer = np.where(active, qx[flow_check_layer, row0:row1, col0:col1], np.nan)
    qy_layer = np.where(active, qy[flow_check_layer, row0:row1, col0:col1], np.nan)

    im = ax.imshow(
        np.ma.masked_invalid(h),
        origin="upper",
        extent=[xmin, xmax, ymin, ymax],
    )

    cs = ax.contour(x, y, h, levels=flow_head_contour_count, linewidths=0.8)
    ax.clabel(cs, fontsize=7, inline=True)

    ax.quiver(
        x[::flow_vector_stride, ::flow_vector_stride],
        y[::flow_vector_stride, ::flow_vector_stride],
        qx_layer[::flow_vector_stride, ::flow_vector_stride],
        qy_layer[::flow_vector_stride, ::flow_vector_stride],
        pivot="middle",
    )

    if len(counties) > 0:
        counties.boundary.plot(ax=ax, linewidth=0.8)

    ax.set_title(title)
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    fig.colorbar(im, ax=ax, shrink=0.75, label="head")

plt.show()

# Direct check on the budget terms GWT reads for intercell transport.
forward_flowja = flowja_values(reverse_forward_budget_file)
reversed_flowja = flowja_values(reverse_budget_file)
flowja_mask = np.isfinite(forward_flowja) & np.isfinite(reversed_flowja) & (np.abs(forward_flowja) > 0)

ratio = reversed_flowja[flowja_mask] / forward_flowja[flowja_mask]
relative_mismatch = np.nanmax(np.abs(forward_flowja[flowja_mask] + reversed_flowja[flowja_mask])) / np.nanmax(np.abs(forward_flowja[flowja_mask]))

print(f"Forward head time: {forward_head_time / 365.25:.2f} years")
print(f"Reversed head time: {reversed_head_time / 365.25:.2f} years")
print(f"Forward budget time: {forward_budget_time / 365.25:.2f} years")
print("Median reversed/FLOW-JA-FACE forward ratio:", np.nanmedian(ratio))
print("Maximum relative |forward + reversed| mismatch:", relative_mismatch)
print("Expected for a clean reversal: median ratio near -1 and mismatch near 0.")


## 18. Background and interpretation

This notebook uses two transport ideas together: **direct groundwater-age simulation** and **reverse transport for capture-zone screening**.

### Groundwater age

Groundwater age estimates how long water has been moving through the modeled aquifer system. It helps show where the model behaves like a younger, faster-connected flow system and where it behaves like an older, slower, or more isolated one.

The age map is not a direct field measurement at every cell. It is a model interpretation built from the simulated flow system, porosity assumptions, transport setup, and boundary conditions. That does not make it unhelpful. It means the map should be read as a regional pattern map rather than as a perfect stopwatch underground.

### Reverse capture

Reverse capture starts at the pumping wells and traces a unit signal backward through the modeled flow system. The resulting concentration is best read as relative connection strength. Higher values indicate stronger modeled connection to the selected pumping locations during the selected time window.

The concentration threshold matters because a transport solution fades outward. A very small nonzero value can cover a large area, especially when numerical dispersion, grid resolution, time stepping, and solver tolerances are involved. In earlier threshold testing, `C > 0` was too broad to be useful as a main planning footprint. The `0.01` and `0.05` footprints were much more stable and told a similar spatial story. This notebook uses `C > 0.05` as the main stronger-signal planning footprint.

### 🌌 A Water Modeling Parable: Salacious Crumb and the Problem of Tiny Signals

Salacious B. Crumb is memorable, noisy, and technically present, but he should not be allowed to run the entire palace. Very small modeled concentrations are similar. They are real outputs from the numerical solution, but they may represent weak signal, numerical spreading, or the fuzzy edge of the calculation rather than a meaningful planning area.

That is why the threshold is not a cosmetic mapping choice. It defines what level of modeled connection is strong enough to include in the main capture-zone footprint. Lower thresholds can still be useful as screening or sensitivity information, but they should not automatically become the public-facing boundary.

### References behind the method

- **Goode (1996)** provides the classic direct-simulation approach for groundwater age.
- **Neupauer and Wilson** and related backward/adjoint transport work provide the conceptual basis for tracing connection backward from a receptor or pumping location.
- **Tosco and Sethi** discuss backward probability and particle-tracking approaches for wellhead-protection-style capture zones, including the importance of how a capture-zone boundary is defined.

The practical message is straightforward: these maps are not final legal boundaries. They are transparent, repeatable model products that help people ask better questions about groundwater connection, travel time, uncertainty, and protection.


## 19. Reading the results together

The maps are most useful when they are read as a sequence rather than as separate products.

Start with the **groundwater-age map**. It gives regional context for the flow system: where the model suggests groundwater is younger, older, faster-connected, or slower-moving.

Then look at the **flow-reversal check**. This figure explains the mechanics of the reverse-capture setup. The head pattern provides familiar hydraulic context, while the arrows show the forward flow direction beside the reversed transport direction used for capture mapping.

Finally, read the **capture-zone maps**. The 5-year zone shows the strongest near-term modeled connection to the pumping wells. The 10- and 25-year zones show how that connected footprint expands as the travel-time window gets longer. The local map brings the same result into a smaller area where roads, boundaries, wells, and community questions can be discussed more directly.

The strongest conclusions are the patterns that survive across these views. The most useful questions often sit near the edges: places where the mapped footprint depends on threshold choice, layer structure, pumping assumptions, or local hydrogeologic detail.

### 🌌 A Water Modeling Parable: Victory Belongs to the Whole System

The end of *Return of the Jedi* is not won by one tool alone. The fleet matters. Luke matters. The Ewoks matter. The shield generator matters. Even the messy, awkward parts of the mission matter.

Groundwater modeling works the same way. A single map rarely gives the whole answer. The age map, the flow-reversal check, and the capture-zone maps each show a different part of the system. Together, they help a group move from “Where is the well?” to “How is this well connected to the aquifer over time?”

### Technical accomplishments

In completing this notebook, you:

1. ✅ Loaded the IISG base groundwater-flow model
2. ✅ Created a separate groundwater-age modeling workspace
3. ✅ Added a GWT model to simulate total groundwater age
4. ✅ Ran the groundwater-age model and mapped the final age field
5. ✅ Built a reverse-capture workflow around selected pumping wells
6. ✅ Reversed the saved flow information used by the transport model
7. ✅ Ran a reverse-GWT model for selected lookback years
8. ✅ Mapped nested 5-, 10-, and 25-year capture-zone footprints
9. ✅ Checked the reversed transport direction using heads, arrows, and budget terms
10. ✅ Interpreted the results as model-based planning information rather than fixed underground borders

### Suggested extensions

- **The Rancor Pit Test:** Change `capture_concentration_threshold` and compare how the capture-zone footprint changes. Which areas remain stable, and which areas only appear at low thresholds?
- **Ewok Local Knowledge:** Adjust the local map window around areas that participants recognize. Local knowledge can help identify whether model patterns match wells, streams, land use, or known geologic transitions.
- **The Shield Generator Scenario:** Compare capture zones before and after changing pumping rates or well locations. Which management choices expand or shrink the connected footprint?
- **Salacious Crumb Filter:** Inspect very low concentration thresholds as diagnostic maps, but resist letting tiny numerical signal define the main planning boundary.
- **Death Star II Briefing Map:** Export the age and capture-zone maps for a meeting handout, then ask participants what patterns are clear, what is surprising, and what needs follow-up analysis.
